# Student Model Evaluation (Llama 3.1 8B + LoRA)

This notebook evaluates the fine-tuned student model on the test set.

## Requirements

- **RAM:** 16GB+ required for CPU inference
- **GPU:** Optional (RTX 3070 Ti 8GB VRAM is insufficient)
- **Model:** `models/sql-llama-8b-lora/` must exist
- **Test data:** `data/curated/test.jsonl` must exist

## Hardware Considerations

- CPU inference will be SLOW (several minutes for 14 examples)
- GPU inference requires more VRAM than RTX 3070 Ti provides
- For faster evaluation, use a cloud GPU (RunPod, Lambda Labs)

## Estimated Time

- CPU-only: ~30-45 minutes for 14 examples
- Cloud GPU (A100): ~1-2 minutes

In [ ]:
# Cell 2: All imports and setupimport jsonimport jsonlinesimport loggingimport timefrom pathlib import Pathimport sys# Add project to pathsys.path.append("..")# Import required modules (these will be used later)from transformers import AutoModelForCausalLM, AutoTokenizerfrom peft import PeftModelimport torch# Setuplogging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')logger = logging.getLogger(__name__)print("=== Student Model Evaluation ===")print("Dependencies: torch, transformers, peft")print("")print("NOTE: Model loading requires ~16GB RAM and will take 2-5 minutes.")print("      Inference will take 30-45 minutes for 14 examples on CPU.")

In [ ]:
# Cell 2: Find project root and setup paths
def find_project_root() -> Path:
    """Find project root by searching for marker files."""
    MARKER_FILES = ["CLAUDE.md", "pyproject.toml", ".git"]
    current_path = Path.cwd()
    
    for parent in [current_path] + list(current_path.parents):
        for marker in MARKER_FILES:
            marker_path = parent / marker
            if marker_path.exists():
                return parent
    
    raise RuntimeError(f"Cannot find project root from {current_path}")

PROJECT_ROOT = find_project_root()

# Constants
BASE_MODEL_ID = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
LORA_PATH = PROJECT_ROOT / "models" / "sql-llama-8b-lora"
TEST_DATA_PATH = PROJECT_ROOT / "data" / "curated" / "test.jsonl"

# Verify paths
if not TEST_DATA_PATH.exists():
    raise FileNotFoundError(f"Test data not found: {TEST_DATA_PATH}")
if not LORA_PATH.exists():
    raise FileNotFoundError(f"LoRA adapter not found: {LORA_PATH}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Base model: {BASE_MODEL_ID}")
print(f"LoRA adapter: {LORA_PATH}")
print(f"Test data: {TEST_DATA_PATH}")
print(f"\nAll paths verified.")

In [ ]:
# Cell 3: Load test datatest_data = []with jsonlines.open(TEST_DATA_PATH) as f:    test_data = list(f)print(f"Loaded {len(test_data)} test examples")print(f"\nSample example:")print(f"  NL:  {test_data[0]['natural_language']}")print(f"  SQL: {test_data[0]['sql']}")

## Load Student ModelThe following cell loads the Llama 3.1 8B model with LoRA adapter.**This will take significant time and RAM:**- Model loading: 2-5 minutes- Inference: 2-3 minutes per example (14 examples = 30-45 minutes total)- RAM usage: ~16GBIf you see memory errors, your system doesn't have enough RAM.

In [ ]:
# Cell 4: Load student model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import gc
# Clear cache before loading
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("Cleared GPU cache")

gc.collect()

print(f"Loading base model: {BASE_MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print("⚠️  Loading model on CPU - this will take 2-5 minutes...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
    low_cpu_mem_usage=True,
    max_memory={0: "1GB", "cpu": "30GB"},
)

print(f"Loading LoRA adapter: {LORA_PATH}")
student_model = PeftModel.from_pretrained(base_model, LORA_PATH)
student_model.eval()

# Get model info
total_params = student_model.num_parameters() / 1e9
print(f"\nStudent model loaded: {total_params:.2f}B parameters")
print(f"Device: {next(student_model.parameters()).device}")
print(f"\n⚠️  Running on CPU - inference will be slow but functional")

## Evaluate Student Model

This cell evaluates the student model on all 14 test examples.

**Estimated time:** 30-45 minutes

The cell shows progress every 2 examples.

In [ ]:
# Cell 5: Evaluate student model
from src.evaluate.benchmark import _build_prompt

predictions = []
t0 = time.perf_counter()

print(f"Evaluating student model on {len(test_data)} examples...")
print("This will take 30-45 minutes on CPU. Please be patient.")

batch_size = 2  # Conservative for CPU

for start in range(0, len(test_data), batch_size):
    batch_prompts = [_build_prompt(ex["natural_language"]) for ex in test_data[start:start + batch_size]]
    inputs = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(student_model.device)
    
    with torch.no_grad():
        outputs = student_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # Decode and strip prompt
    prompt_lengths = inputs["input_ids"].shape[1]
    for output in outputs:
        generated = output[prompt_lengths:]
        decoded = tokenizer.decode(generated, skip_special_tokens=True)
        sql = decoded.strip().split("\n")[0].strip().rstrip(";")
        predictions.append(sql)
    
    # Clear cache after each batch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    if (start + batch_size) % 2 == 0 or start + batch_size >= len(test_data):
        print(f"  Progress: {min(start + batch_size, len(test_data))}/{len(test_data)} examples")

elapsed = time.perf_counter() - t0
print(f"\nStudent evaluation complete: {elapsed:.1f}s")
print(f"  Throughput: {len(test_data)/elapsed:.3f} examples/sec")
print(f"  Cost: $0.0000 (local inference)")
print(f"  Avg latency: {elapsed/len(test_data):.3f}s per query")

## Save Results

Save the student predictions to disk for later comparison.

In [ ]:
# Cell 6: Save results
import json
from pathlib import Path
import time

results = {
    "metadata": {
        "test_examples": len(test_data),
        "student_model": "Meta-Llama-3.1-8B + LoRA",
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    },
    "student": {
        "predictions": predictions,
        "elapsed_seconds": round(elapsed, 2),
        "training_cost_usd": 0.50  # One-time training cost
    }
}

output_path = Path("tasks/sql_generation/student_results.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Student results saved to: {output_path}")
print(f"\nFirst 3 predictions:")
for i in range(3):
    print(f"{i+1}. {predictions[i]}")

## Next Steps

1. Results saved to `tasks/sql_generation/student_results.json`
2. To compare with teacher model, run `10_comparison_from_results.ipynb`
3. For faster evaluation in the future, use a cloud GPU (RunPod, Lambda Labs)

**Cloud GPU Recommendations:**
- RunPod: A100 40GB (~$0.40-0.80/hour)
- Lambda Labs: A100 80GB (~$1.19/hour)

**Local upgrade option:**
- Add more RAM (32GB+) for CPU inference
- GPU with 16GB+ VRAM (RTX 3090, RTX 4090, A6000, etc.)